# Bayesian inference 101

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sthsci/Orca/blob/main/notebooks/01_bayesian_inference_101.ipynb)

**Teaching notebook.** Use a coin-toss example to connect priors, likelihoods, posteriors, credible intervals, marginal likelihoods, and Bayes factors.

Run the cells from top to bottom. Values collected near the start of each notebook are safe places to experiment. Bayesian SMC fitting is deliberately disabled by default in the analysis notebooks because it can take several minutes; set `RUN_INFERENCE = True` when the data checks and descriptive plots look right.

Use synthetic or approved anonymised data only. Do not upload names, clinical metadata, raw microscopy, or a donor key that could identify participants.


## 1. The Bayesian update

Bayes' theorem combines a prior with evidence from the data:

$$p(\theta\mid y)=\frac{p(y\mid\theta)p(\theta)}{p(y)}.$$

- $p(\theta)$ is the **prior**.
- $p(y\mid\theta)$ is the **likelihood**.
- $p(\theta\mid y)$ is the **posterior**.
- $p(y)$ is the **marginal likelihood**, the average likelihood under a model's prior.

Edit the values below. A Beta prior and Binomial likelihood give a Beta posterior exactly, so no sampler is needed.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import betaln, gammaln
from scipy.stats import beta

PRIOR_A = 2.0
PRIOR_B = 2.0
TOSSES = 20
HEADS = 14

if not (0 <= HEADS <= TOSSES):
    raise ValueError("HEADS must be between zero and TOSSES.")
if PRIOR_A <= 0 or PRIOR_B <= 0:
    raise ValueError("Beta prior parameters must be positive.")

POSTERIOR_A = PRIOR_A + HEADS
POSTERIOR_B = PRIOR_B + TOSSES - HEADS


In [ ]:
theta = np.linspace(0.001, 0.999, 600)
prior_density = beta.pdf(theta, PRIOR_A, PRIOR_B)
likelihood = theta**HEADS * (1 - theta) ** (TOSSES - HEADS)
likelihood /= np.trapz(likelihood, theta)
posterior_density = beta.pdf(theta, POSTERIOR_A, POSTERIOR_B)

interval = beta.ppf([0.025, 0.975], POSTERIOR_A, POSTERIOR_B)
posterior_mean = POSTERIOR_A / (POSTERIOR_A + POSTERIOR_B)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(theta, prior_density, label="Prior", linewidth=2)
ax.plot(theta, likelihood, label="Scaled likelihood", linewidth=2)
ax.plot(theta, posterior_density, label="Posterior", linewidth=3)
ax.axvspan(*interval, color="#34C759", alpha=0.15, label="95% credible interval")
ax.axvline(posterior_mean, color="#304B3D", linestyle="--")
ax.set(xlabel="Probability of heads, θ", ylabel="Density")
ax.legend(frameon=False)
plt.show()

print(f"Posterior mean: {posterior_mean:.3f}")
print(f"95% credible interval: [{interval[0]:.3f}, {interval[1]:.3f}]")


## 2. Compare models with a Bayes factor

Let $\mathcal M_0$ fix $\theta=0.5$. Let $\mathcal M_1$ allow $\theta$ to vary under the chosen Beta prior. Their marginal likelihoods average over all parameter values each model permits.

$$\mathrm{BF}_{10}=\frac{p(y\mid\mathcal M_1)}{p(y\mid\mathcal M_0)}.$$

A value above one favours $\mathcal M_1$ relative to $\mathcal M_0$; a value below one favours $\mathcal M_0$. This is relative evidence, not the probability that either model is true.


In [ ]:
log_choose = gammaln(TOSSES + 1) - gammaln(HEADS + 1) - gammaln(TOSSES - HEADS + 1)
log_evidence_fixed = log_choose + TOSSES * math.log(0.5)
log_evidence_flexible = (
    log_choose
    + betaln(HEADS + PRIOR_A, TOSSES - HEADS + PRIOR_B)
    - betaln(PRIOR_A, PRIOR_B)
)
bf_10 = math.exp(log_evidence_flexible - log_evidence_fixed)

print(f"log p(data | fixed fair coin): {log_evidence_fixed:.3f}")
print(f"log p(data | flexible coin):   {log_evidence_flexible:.3f}")
print(f"BF_10: {bf_10:.3f}")


## 3. Why ORCA uses SMC

The coin posterior is available in closed form. ORCA's hierarchical and trajectory models are not, so it represents the posterior with samples.

- **MCMC** constructs a correlated chain whose long-run distribution is the posterior.
- **Sequential Monte Carlo (SMC)** moves a population of particles from the prior toward the posterior through intermediate distributions.
- ORCA uses PyMC SMC because the same run supplies posterior particles and an estimate of the marginal likelihood used for Bayes factors.

More particles and independent chains usually improve stability but cost more computation. Bayes factors are sensitive to prior choices, so report priors and check whether the scientific conclusion survives reasonable alternatives.

**Try next:** change `HEADS`, `TOSSES`, `PRIOR_A`, and `PRIOR_B`; rerun the notebook; then continue to the event-count model tutorial.
